# Customer Analytics: RFM Analysis & Customer Segmentation

Notebook ini bertujuan untuk melakukan analisis segmentasi pelanggan menggunakan metode **RFM (Recency, Frequency, Monetary)** langsung dari database PostgreSQL (`smart_retail`).

**Tujuan Analysis:**
1. Mengambil data gabungan transaksi dan profil pelanggan.
2. Menghitung nilai Recency, Frequency, dan Monetary per pelanggan.
3. Melakukan kuantil scoring (1-5) untuk setiap atribut RFM.
4. Mengelompokkan pelanggan ke dalam segmen perilaku (Champions, Loyal, At Risk, dsb).
5. Menganalisis profil pelanggan bernilai tinggi dan ringkasan segmen.

## 1. Load Customer Transaction Data

Mengambil seluruh data transaksi yang telah terhubung dengan data master pelanggan langsung dari tabel `sales_transactions` dan `customers` di PostgreSQL.

In [2]:
import sys
from pathlib import Path
import pandas as pd

# Set path root project untuk mengimpor modul kustom etl
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

from etl.db_connection import get_engine

# Inisialisasi koneksi database
engine = get_engine()

# Query SQL untuk mengambil data transaksi beserta informasi profil customer
query = """
SELECT
    s.transaction_id,
    s.transaction_date,
    s.customer_id,
    s.transaction_qty,
    s.unit_price,
    s.transaction_qty * s.unit_price AS revenue,
    c.customer_name,
    c.gender,
    c.birth_year,
    c.city,
    c.join_date,
    c.customer_segment
FROM sales_transactions s
JOIN customers c
    ON s.customer_id = c.customer_id;
"""

# Read data ke dalam pandas DataFrame
customer_sales = pd.read_sql(query, engine)

# Tampilkan ringkasan dataset
print("Rows:", len(customer_sales))
print("Unique Customers:", customer_sales["customer_id"].nunique())

Rows: 149116
Unique Customers: 17341


## 2. Kalkulasi Metrik RFM (Recency, Frequency, Monetary)

Menghitung 3 metrik utama untuk setiap `customer_id`:
* **Recency (R):** Jumlah hari sejak transaksi terakhir pelanggan hingga `reference_date` (1 hari setelah tanggal transaksi paling akhir di dataset).
* **Frequency (F):** Jumlah transaksi unik yang dilakukan pelanggan.
* **Monetary (M):** Total nilai transaksi (revenue) yang dihasilkan oleh pelanggan.

In [3]:
# Tentukan reference date (H+1 dari tanggal transaksi paling baru di dataset)
reference_date = customer_sales["transaction_date"].max() + pd.Timedelta(days=1)

# Hitung metrik RFM menggunakan agregasi groupby
rfm = (
    customer_sales.groupby("customer_id")
    .agg(
        recency=(
            "transaction_date",
            lambda x: (reference_date - x.max()).days,  # Selisih hari ke transaksi terakhir
        ),
        frequency=("transaction_id", "nunique"),  # Total transaksi unik
        monetary=("revenue", "sum"),  # Total belanjaan
    )
    .reset_index()
)

# Tampilkan 5 sampel data pertama hasil kalkulasi RFM
rfm.head()

,customer_id,recency,frequency,monetary
0,1,16,12,45.85
1,3,26,4,15.95
2,4,7,23,128.60
3,6,77,1,5.00
4,7,45,1,5.10


## 3. Pembentukan Skor RFM (Quantile Scoring 1-5)

Membagi pelanggan ke dalam 5 kelompok kuantil (Skor 1 sampai 5) berdasarkan masing-masing metrik:
* **R_score:** Diberi skor `5` (terbaik/terbaru) hingga `1` (terlama).
* **F_score:** Diberi skor `1` (frekuensi terendah) hingga `5` (frekuensi tertinggi).
* **M_score:** Diberi skor `1` (monetary terendah) hingga `5` (monetary tertinggi).

In [4]:
# Skor Recency (Nilai recency lebih kecil = transaksi lebih baru = Skor R lebih tinggi)
rfm["R_score"] = pd.qcut(
    rfm["recency"], 5, labels=[5, 4, 3, 2, 1], duplicates="drop"
).astype(int)

# Skor Frequency (Gunakan rank method 'first' untuk menangani duplikasi nilai kuantil)
rfm["F_score"] = pd.qcut(
    rfm["frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]
).astype(int)

# Skor Monetary
rfm["M_score"] = pd.qcut(
    rfm["monetary"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]
).astype(int)

# Penggabungan string skor menjadi RFM Combined Score (contoh: '555', '111')
rfm["rfm_score"] = (
    rfm["R_score"].astype(str)
    + rfm["F_score"].astype(str)
    + rfm["M_score"].astype(str)
)

# Tampilkan sampel data dengan kolom scoring baru
rfm.head()

,customer_id,recency,frequency,monetary,R_score,F_score,M_score,rfm_score
0,1,16,12,45.85,3,4,4,344
1,3,26,4,15.95,2,2,2,222
2,4,7,23,128.60,4,5,5,455
3,6,77,1,5.00,1,1,1,111
4,7,45,1,5.10,2,1,1,211


## 4. Segmentasi Pelanggan Berdasarkan Aturan RFM

Mengelompokkan pelanggan ke dalam segmen bisnis bisnis berbasis aturan (*rule-based logic*) menggunakan gabungan skor R, F, dan M.

In [5]:
# Fungsi logika pengelompokan segmen pelanggan
def segment_customer(row):
    r = row["R_score"]
    f = row["F_score"]
    m = row["M_score"]

    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"

    if r >= 4 and f >= 3:
        return "Loyal Customers"

    if r >= 4 and f <= 2:
        return "New / Promising"

    if r <= 2 and f >= 4 and m >= 4:
        return "At Risk High Value"

    if r <= 2 and f >= 3:
        return "At Risk"

    if r <= 2 and f <= 2:
        return "Lost / Hibernating"

    return "Potential"


# Terapkan fungsi segmentasi ke setiap baris data
rfm["rfm_segment"] = rfm.apply(segment_customer, axis=1)

# Tampilkan jumlah pelanggan per segmen hasil pengelompokan
rfm["rfm_segment"].value_counts()

rfm_segment
Lost / Hibernating    4584
Champions             4504
Potential             3201
At Risk               1612
Loyal Customers       1475
New / Promising       1258
At Risk High Value     707
Name: count, dtype: int64

## 5. Penggabungan Segmen RFM dengan Profil Pelanggan

Menggabungkan segmen perilaku RFM yang baru dibentuk dengan data demografi asli pelanggan (`customer_name`, `gender`, `city`, `customer_segment`).

In [6]:
# Ambil data atribut unik profil pelanggan dari DataFrame utama
customer_profile = customer_sales[
    [
        "customer_id",
        "customer_name",
        "gender",
        "birth_year",
        "city",
        "customer_segment",
    ]
].drop_duplicates("customer_id")

# Merge profil pelanggan dengan metrik & segmen RFM
customer_analysis = customer_profile.merge(rfm, on="customer_id", how="inner")

# Tampilkan sampel dataset analisis pelanggan akhir
customer_analysis.head()

,customer_id,customer_name,gender,birth_year,city,customer_segment,recency,frequency,monetary,R_score,F_score,M_score,rfm_score,rfm_segment
0,14928,William White,Female,1989,Lower Manhattan,Young Professional,57,3,13.50,1,2,2,122,Lost / Hibernating
1,8755,John Williams,Female,1981,Lower Manhattan,Family,10,8,71.70,4,4,5,445,Champions
2,17344,Michael Taylor,Male,1998,Lower Manhattan,Young Professional,2,17,78.90,5,5,5,555,Champions
3,12259,William Jones,Female,1965,Lower Manhattan,Mature,1,37,145.35,5,5,5,555,Champions
4,4286,Grace Williams,Male,1965,Lower Manhattan,Mature,29,24,114.10,2,5,5,255,At Risk High Value


## 6. Analisis Pelanggan Bernilai Tinggi (Top 10 Valuable Customers)

Identifikasi 10 pelanggan teratas berdasarkan total kontribusi pendapatan belanja (*monetary*).

In [7]:
# Urutkan pelanggan berdasarkan kolom monetary secara descending
top_10_customers = customer_analysis.sort_values(
    "monetary", ascending=False
).head(10)

top_10_customers

,customer_id,customer_name,gender,birth_year,city,customer_segment,recency,frequency,monetary,R_score,F_score,M_score,rfm_score,rfm_segment
112,11590,Robert Brown,Male,1991,Lower Manhattan,Young Professional,1,360,1748.47,5,5,5,555,Champions
718,8092,William Smith,Female,1988,Hell's Kitchen,Family,3,295,1672.65,5,5,5,555,Champions
1238,4223,William Miller,Female,1989,Astoria,Young Professional,2,176,795.85,5,5,5,555,Champions
1994,5184,James Thomas,Male,1968,Hell's Kitchen,Mature,1,152,706.80,5,5,5,555,Champions
924,6503,James Taylor,Male,1985,Hell's Kitchen,Family,3,141,702.50,5,5,5,555,Champions
624,2864,Emma Johnson,Male,1999,Lower Manhattan,Student,1,136,697.13,5,5,5,555,Champions
394,11162,Michael Thomas,Male,1980,Lower Manhattan,Family,1,158,693.24,5,5,5,555,Champions
1505,1770,Emma White,Female,1990,Hell's Kitchen,Young Professional,1,141,670.75,5,5,5,555,Champions
954,4619,Olivia Davis,Male,1969,Astoria,Mature,1,112,635.58,5,5,5,555,Champions
166,19909,Thomas Martin,Female,1977,Hell's Kitchen,Family,1,119,632.95,5,5,5,555,Champions


## 7. Ringkasan Eksekutif Setiap Segmen RFM

Agregasi metrik bisnis utama (jumlah pelanggan, rata-rata recency, rata-rata frekuensi, total revenue, dan rata-rata revenue) per segmen RFM.

In [8]:
# Agregasi data berdasarkan rfm_segment
segment_summary = (
    customer_analysis.groupby("rfm_segment")
    .agg(
        customers=("customer_id", "count"),
        avg_recency=("recency", "mean"),
        avg_frequency=("frequency", "mean"),
        total_revenue=("monetary", "sum"),
        avg_revenue=("monetary", "mean"),
    )
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)

# Tampilkan tabel ringkasan segmen
segment_summary

,rfm_segment,customers,avg_recency,avg_frequency,total_revenue,avg_revenue
2,Champions,4504,4.824156,20.508659,434345.05,96.435402
6,Potential,3201,15.946267,6.890034,104668.08,32.698557
3,Lost / Hibernating,4584,67.021161,1.872382,39987.24,8.723220
0,At Risk,1612,41.500000,5.033499,36933.37,22.911520
4,Loyal Customers,1475,5.791186,5.453559,35659.89,24.176197
1,At Risk High Value,707,32.248939,10.083451,33703.99,47.671839
5,New / Promising,1258,5.881558,2.241653,13514.71,10.743013
